# Decision trees (CART (Classification And Regression Trees))

_Machine Learning_

**Ask yes/no questions, split the data, repeat. Easy to read.**

A decision tree classifies by asking a sequence of simple questions.

     Each question splits the data into two groups.

---

This notebook is a practice scaffold for the **Decision trees (CART (Classification And Regression Trees))** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — scikit-learn

### Step 1 — Make a labelled dataset and split it

We generate 400 synthetic examples with 5 features (3 of them genuinely informative). Then we hold back a quarter of the rows as a **test set** so we can measure accuracy on data the tree never saw during training. Keeping training and testing separate is what stops us from fooling ourselves with memorised answers.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

# 400 examples, 5 features, of which 3 actually carry signal.
X, y = make_classification(
    n_samples=400,
    n_features=5,
    n_informative=3,
    random_state=0,
)

# Hold back 25% of the rows to test on.
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0)

print("train rows:", Xtr.shape[0], " test rows:", Xte.shape[0])

### Step 2 — Grow a shallow tree and score it

A `DecisionTreeClassifier` repeatedly splits the data on the feature that best separates the classes. We cap it at `max_depth=3` so it stays small and readable instead of memorising the training set. We then score it on the untouched test rows to see how well it generalises.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Limit the depth so the tree stays simple and does not overfit.
tree = DecisionTreeClassifier(max_depth=3, random_state=0).fit(Xtr, ytr)

test_accuracy = tree.score(Xte, yte)
print("test accuracy:", round(test_accuracy, 3))

### Step 3 — Inspect what the tree learned

Each feature gets an **importance** score: how much that feature reduced impurity across all the splits that used it. We can also print the tree's top levels as plain text to read the actual yes/no questions it asks. Together these tell us *which* measurements the tree leaned on and *how* it branches.

In [ ]:
from sklearn.tree import export_text

# Importance = total impurity reduction contributed by each feature.
importances = np.round(tree.feature_importances_, 3)
print("feature importances:", importances)

# A plain-text view of the first couple of decision levels.
tree_text = export_text(tree, max_depth=2).strip()
print(tree_text[:200])

## Visualize the data & results

_Which of the 30 tumor measurements does a decision tree rely on most to call malignant vs benign?_

### Step 1 — Train a tree on the real breast-cancer scans

Now we swap the synthetic data for 569 real tumour scans, each with 30 measurements. We split off a test set and fit the same depth-3 tree, exactly as before — only the data has changed. This gives us a model whose feature importances reflect real diagnostic signal.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

bc = load_breast_cancer()
Xtr, Xte, ytr, yte = train_test_split(bc.data, bc.target, test_size=0.25, random_state=0)

tree = DecisionTreeClassifier(max_depth=3, random_state=0).fit(Xtr, ytr)

### Step 2 — Rank the features and plot the top 5

We read off each feature's importance, sort them, and keep the five biggest contributors. A bar chart of those five shows at a glance which measurements the tree trusts most when calling a tumour malignant or benign. Most of the 30 features barely matter — a handful do almost all the work.

In [ ]:
# Impurity reduction contributed by each of the 30 features.
imp = tree.feature_importances_

# Indices of the 5 most important features, highest first.
order = np.argsort(imp)[::-1][:5]
top_names = [bc.feature_names[i] for i in order]
top_values = imp[order]

plt.bar(top_names, top_values, color="#4ea1ff")
plt.xticks(rotation=30, ha="right")
plt.ylabel("importance")
plt.title("Decision-tree feature importances")
plt.tight_layout()
plt.show()